# Transformer


## Self-Attention

Implementing the attention mechanism.
We will be using torch and numpy to speed up the matrix multiplications instead of directly using the transformer implementation.

In the image below, the design of one Encoder Block is given. We want you to set up this Block. We will use the implementation of the Self-Attention (doesn't have to be multi-head) and build the Add & Norm and Feed Forward layers on top of it. Add & Norm and the Feed Forward should be implementations by PyTorch or else. We will be using our own Self-Attention function.

* We will be showing that the model block worked, by forwarding a randomly initialized tensor through it once. Printing the values of the Random input tensor, the output tensor and the Q,K and V matrices.

In [ ]:
from IPython.display import Image
Image(url="https://www.researchgate.net/publication/334288604/figure/fig1/AS:778232232148992@1562556431066/The-Transformer-encoder-structure.ppm", height=300)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class SelfAttention(nn.Module):
    """
    Single-head self-attention implemented from scratch.
    Input:  x of shape (B, T, D)
    Output: out of shape (B, T, D)
    Also returns Q, K, V for printing.
    """
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        # x: (B, T, D)
        Q = self.W_q(x)  # (B, T, D)
        K = self.W_k(x)  # (B, T, D)
        V = self.W_v(x)  # (B, T, D)

        # Attention scores: (B, T, T)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model)

        # Softmax over keys dimension
        attn = torch.softmax(scores, dim=-1)  # (B, T, T)

        # Weighted sum: (B, T, D)
        out = torch.matmul(attn, V)
        return out, Q, K, V


class TransformerEncoderBlock(nn.Module):
    """
    Encoder block: Self-Attention -> Add&Norm -> FFN -> Add&Norm
    Self-Attention is custom; FFN and LayerNorm are from PyTorch.
    """
    def __init__(self, d_model, d_ff=128, dropout=0.1):
        super().__init__()
        self.sa = SelfAttention(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, Q, K, V = self.sa(x)
        x = self.ln1(x + self.drop(attn_out))      # Add & Norm
        ff_out = self.ff(x)
        x = self.ln2(x + self.drop(ff_out))        # Add & Norm
        return x, Q, K, V


# ---- Demonstration forward pass ----
B, T, D = 2, 5, 16  # batch, sequence length, embedding dim
x = torch.randn(B, T, D)

block = TransformerEncoderBlock(d_model=D, d_ff=64, dropout=0.0)
y, Q, K, V = block(x)

print("Random input tensor x:\n", x)
print("\nQ matrix:\n", Q)
print("\nK matrix:\n", K)
print("\nV matrix:\n", V)
print("\nOutput tensor y:\n", y)


Random input tensor x:
 tensor([[[-1.1258e+00, -1.1524e+00, -2.5058e-01, -4.3388e-01,  8.4871e-01,
           6.9201e-01, -3.1601e-01, -2.1152e+00,  3.2227e-01, -1.2633e+00,
           3.4998e-01,  3.0813e-01,  1.1984e-01,  1.2377e+00,  1.1168e+00,
          -2.4728e-01],
         [-1.3527e+00, -1.6959e+00,  5.6665e-01,  7.9351e-01,  5.9884e-01,
          -1.5551e+00, -3.4136e-01,  1.8530e+00,  7.5019e-01, -5.8550e-01,
          -1.7340e-01,  1.8348e-01,  1.3894e+00,  1.5863e+00,  9.4630e-01,
          -8.4368e-01],
         [-6.1358e-01,  3.1593e-02, -4.9268e-01,  2.4841e-01,  4.3970e-01,
           1.1241e-01,  6.4079e-01,  4.4116e-01, -1.0231e-01,  7.9244e-01,
          -2.8967e-01,  5.2507e-02,  5.2286e-01,  2.3022e+00, -1.4689e+00,
          -1.5867e+00],
         [-6.7309e-01,  8.7283e-01,  1.0554e+00,  1.7784e-01, -2.3034e-01,
          -3.9175e-01,  5.4329e-01, -3.9516e-01, -4.4622e-01,  7.4402e-01,
           1.5210e+00,  3.4105e+00, -1.5312e+00, -1.2341e+00,  1.8197e+00,
    

### Use your own Transformer Block

* Chain 3 of the transformer blocks to set up a model. Put 1 fully connected layer head on top.
* Train the model on the MNIST dataset for image classification.
* Reporting the test accuracy after training.

Are we able to make our own attention work? :)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Reuse TransformerEncoderBlock from Task 11.1.1 above.

class TinyVisionTransformer(nn.Module):
    """
    Simple model for MNIST:
    - x: (B, 1, 28, 28)
    - tokens: treat each row as a token -> (B, 28, 28)
    - project to d_model
    - 3 encoder blocks
    - mean pool over tokens
    - FC head to 10 classes
    """
    def __init__(self, d_model=64, d_ff=128, n_blocks=3, dropout=0.1, n_classes=10):
        super().__init__()
        self.d_model = d_model
        self.in_proj = nn.Linear(28, d_model)  # each token has 28 features (a row)
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(d_model=d_model, d_ff=d_ff, dropout=dropout)
            for _ in range(n_blocks)
        ])
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        # x: (B, 1, 28, 28)
        x = x.squeeze(1)            # (B, 28, 28)
        x = self.in_proj(x)         # (B, 28, d_model)
        for blk in self.blocks:
            x, _, _, _ = blk(x)     # ignore Q/K/V during training
        x = x.mean(dim=1)           # (B, d_model) global average pooling over tokens
        logits = self.head(x)       # (B, 10)
        return logits


def accuracy(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.numel()
    return correct / total


# ---- Data ----
transform = transforms.ToTensor()
train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# ---- Train ----
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TinyVisionTransformer(d_model=64, d_ff=128, n_blocks=3, dropout=0.1).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

epochs = 10

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        opt.step()
        running_loss += loss.item()

    test_acc = accuracy(model, test_loader, device)
    print(f"Epoch {epoch}/{epochs} | loss={running_loss/len(train_loader):.4f} | test_acc={test_acc:.4f}")

# ---- Final report (required) ----
final_test_acc = accuracy(model, test_loader, device)
print("Final test accuracy:", final_test_acc)


Epoch 1/10 | loss=0.8985 | test_acc=0.8273
Epoch 2/10 | loss=0.4693 | test_acc=0.8660
Epoch 3/10 | loss=0.3906 | test_acc=0.8840
Epoch 4/10 | loss=0.3430 | test_acc=0.8890
Epoch 5/10 | loss=0.3149 | test_acc=0.8968
Epoch 6/10 | loss=0.2902 | test_acc=0.9062
Epoch 7/10 | loss=0.2729 | test_acc=0.9040
Epoch 8/10 | loss=0.2625 | test_acc=0.9149
Epoch 9/10 | loss=0.2418 | test_acc=0.9060
Epoch 10/10 | loss=0.2395 | test_acc=0.9181
Final test accuracy: 0.9181
